# Change Data Feed (CDF) → Delta analytics

Capstone requirement: **enable CDF** on the marine Delta table, **readChangeFeed**, write analytics Delta.

**Prereq:** run `run_marine_pipeline.ipynb` once so `coastal_ops_marine_conditions` exists.

This notebook:
1. `ALTER TABLE … SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`
2. Reads CDF with `readChangeFeed=true`
3. Writes UC tables:
   - `coastal_ops_marine_cdf_changes` (bronze)
   - `coastal_ops_marine_cdf_analytics` (gold)

Attach Serverless or a Spark cluster, then Run all.

In [ ]:
import importlib.util
from pathlib import Path

script = None
for root in [Path.cwd(), Path.cwd().parent, *list(Path.cwd().parents)[:5]]:
    for hit in (
        root / "notebooks" / "cdf_marine_analytics_to_delta.py",
        root / "cdf_marine_analytics_to_delta.py",
    ):
        if hit.exists():
            script = hit
            break
    if script is not None:
        break

if script is None:
    raise FileNotFoundError("Could not find cdf_marine_analytics_to_delta.py")

print("Loading", script)
spec = importlib.util.spec_from_file_location("cdf_marine_analytics_to_delta", script)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
summary = mod.main()
summary

## Evidence for graders

1. CDF property on the source table  
2. Gold analytics from the change feed

In [ ]:
catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
schema = spark.sql("SELECT current_schema()").collect()[0][0]
source = f"{catalog}.{schema}.coastal_ops_marine_conditions"
gold = f"{catalog}.{schema}.coastal_ops_marine_cdf_analytics"
bronze = f"{catalog}.{schema}.coastal_ops_marine_cdf_changes"

print("Source:", source)
display(spark.sql(f"SHOW TBLPROPERTIES {source}"))

In [ ]:
print("CDF bronze:", bronze)
display(spark.table(bronze))

In [ ]:
print("CDF gold analytics:", gold)
display(spark.table(gold))